# RT-V1 — chạy trực tiếp một cell
Cài các thư viện theo hướng dẫn trước. Sửa DATA_INPUT, MODEL_DIR, OUTPUT_DIR trong cell. Không cần file script Python. Nhãn bổ sung là AI silver, không phải human gold.

In [ ]:
# CHỈ SỬA ĐƯỜNG DẪN VÀ TÙY CHỌN Ở ĐÂY, SAU ĐÓ CHẠY CẢ CELL.
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # đặt trước khi import torch
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ZIP RT-V1 hoặc thư mục đã giải nén chứa MANIFEST.json và các parquet.
DATA_INPUT = Path("/kaggle/input/TEN_DATASET_DU_LIEU/retrieval_finetune_v1.zip")
# THƯ MỤC model E2 chứa config.json, trọng số và tokenizer.
MODEL_DIR = Path("/kaggle/input/TEN_DATASET_MODEL/best_model")
OUTPUT_DIR = Path("/kaggle/working/rt_v1_silver")

TRAIN_FILE = "train.parquet"  # đối chứng: "gold_train.parquet"
EXPERIMENT_NAME = "RT-V1"  # Ví dụ: "RT-V2-balanced" cho ZIP cân bằng nhãn.
EPOCHS = 2
MAX_STEPS = -1  # -1: chạy đầy đủ; 2: chỉ smoke test

"""RT-V1: continue E2 on gold train + optional AI silver; evaluate DEV only."""
from types import SimpleNamespace
import zipfile
import hashlib
import json
import os
from pathlib import Path

LABELS = ["SUPPORTED", "REFUTED", "NOT_ENOUGH_INFO"]


def checksum(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def run_finetune():
    args = SimpleNamespace(
        data_dir=DATA_INPUT, model_dir=MODEL_DIR, output_dir=OUTPUT_DIR,
        train_file=TRAIN_FILE, experiment=EXPERIMENT_NAME, epochs=EPOCHS, max_steps=MAX_STEPS,
    )
    if args.output_dir.exists() and any(args.output_dir.iterdir()):
        raise FileExistsError("Use a fresh output directory to preserve prior results")
    if args.epochs < 1 or args.max_steps == 0 or args.max_steps < -1:
        raise ValueError("Invalid epochs/max-steps")

    import numpy as np
    import pandas as pd
    import py_vncorenlp
    import torch
    import torch.nn.functional as F
    import transformers
    from datasets import Dataset
    from sklearn.metrics import accuracy_score, classification_report, f1_score
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, EarlyStoppingCallback, Trainer, TrainingArguments, set_seed

    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU before training")
    if args.data_dir.suffix.lower() == ".zip":
        extraction = args.output_dir / "input_data"
        with zipfile.ZipFile(args.data_dir) as archive:
            for member in archive.namelist():
                if not (extraction / member).resolve().is_relative_to(extraction.resolve()):
                    raise ValueError("Unsafe ZIP member")
            extraction.mkdir(parents=True, exist_ok=False)
            archive.extractall(extraction)
        args.data_dir = extraction
    manifest_path = args.data_dir / "MANIFEST.json"
    manifest = json.loads(manifest_path.read_text())
    for name, expected in manifest["files"].items():
        path = (args.data_dir / name).resolve()
        if path.parent != args.data_dir.resolve() or checksum(path) != expected:
            raise ValueError(f"Input checksum/path mismatch: {name}")
    train = pd.read_parquet(args.data_dir / args.train_file)
    dev = pd.read_parquet(args.data_dir / "dev.parquet")
    if set(train.Url) & set(dev.Url):
        raise ValueError("Train/dev URL overlap")
    if not train.sample_weight.isin([0.5, 1.0]).all() or not train.labels.isin([0, 1, 2]).all() or not dev.labels.isin([0, 1, 2]).all():
        raise ValueError("Invalid labels or weights")
    expected_mapping = {str(i): label for i, label in enumerate(LABELS)}
    model_config = json.loads((args.model_dir / "config.json").read_text())
    checkpoint_label_mapping = model_config.get("id2label")
    if checkpoint_label_mapping != expected_mapping:
        print(
            "Warning: checkpoint id2label metadata differs. "
            "Training will use the fixed label order: "
            "0=SUPPORTED, 1=REFUTED, 2=NOT_ENOUGH_INFO."
        )
    set_seed(42)
    args.output_dir.mkdir(parents=True, exist_ok=True)
    segment_dir = (args.output_dir / "vncorenlp").resolve()
    segment_dir.mkdir(exist_ok=True)
    if not (segment_dir / "VnCoreNLP-1.2.jar").exists():
        py_vncorenlp.download_model(save_dir=str(segment_dir))
    segmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(segment_dir))
    tokenizer = AutoTokenizer.from_pretrained(str(args.model_dir), use_fast=False, local_files_only=True)
    truncation = {}

    def encode(frame, name):
        claims = [" ".join(segmenter.word_segment(str(s))) for s in frame.Statement]
        passages = [" ".join(segmenter.word_segment(str(s))) for s in frame.Evidence]
        full = tokenizer(claims, passages, truncation=False)
        truncation[name] = sum(len(ids) > 256 for ids in full["input_ids"])
        encoded = tokenizer(claims, passages, truncation=True, max_length=256)
        encoded["labels"] = frame.labels.astype(int).tolist()
        encoded["sample_weight"] = frame.sample_weight.tolist() if "sample_weight" in frame else [1.0] * len(frame)
        return Dataset.from_dict(encoded)

    train_data, dev_data = encode(train, "train"), encode(dev, "dev")
    model = AutoModelForSequenceClassification.from_pretrained(str(args.model_dir), local_files_only=True)
    model.config.id2label = dict(enumerate(LABELS))
    model.config.label2id = {label: idx for idx, label in enumerate(LABELS)}
    padding = DataCollatorWithPadding(tokenizer)

    def collate(features):
        weights = [f["sample_weight"] for f in features]
        batch = padding([{k: v for k, v in f.items() if k != "sample_weight"} for f in features])
        batch["sample_weight"] = torch.tensor(weights, dtype=torch.float32)
        return batch

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            inputs = dict(inputs)
            weights = inputs.pop("sample_weight")
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            losses = F.cross_entropy(outputs.logits.float(), labels, reduction="none")
            loss = (losses * weights).sum() / weights.sum()
            return (loss, outputs) if return_outputs else loss

    def metrics(pred):
        guesses = np.argmax(pred.predictions, axis=-1)
        return {"accuracy": float(accuracy_score(pred.label_ids, guesses)),
                "macro_f1": float(f1_score(pred.label_ids, guesses, labels=[0, 1, 2], average="macro", zero_division=0))}

    training_args = TrainingArguments(
        output_dir=str(args.output_dir / "checkpoints"), learning_rate=1e-5,
        per_device_train_batch_size=8, per_device_eval_batch_size=16, gradient_accumulation_steps=2,
        num_train_epochs=args.epochs, max_steps=args.max_steps, weight_decay=0.01, warmup_ratio=0.1,
        fp16=True, eval_strategy="epoch", save_strategy="epoch", logging_steps=25,
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        save_total_limit=2, seed=42, data_seed=42, report_to="none", remove_unused_columns=False,
    )
    trainer = WeightedTrainer(model=model, args=training_args, train_dataset=train_data, eval_dataset=dev_data,
                              data_collator=collate, compute_metrics=metrics,
                              callbacks=[EarlyStoppingCallback(early_stopping_patience=1)])
    before = trainer.evaluate(metric_key_prefix="before_dev")
    trainer.train()
    predicted = trainer.predict(dev_data, metric_key_prefix="dev")
    guesses = np.argmax(predicted.predictions, axis=-1)
    destination = args.output_dir / "best_model"
    trainer.save_model(str(destination))
    tokenizer.save_pretrained(str(destination))
    rows = dev[["Statement", "Evidence", "Url", "labels"]].copy()
    rows["prediction"] = guesses
    for i, name in enumerate(LABELS):
        rows[f"logit_{name}"] = predicted.predictions[:, i]
    rows.to_csv(args.output_dir / "dev_predictions.csv", index=False)
    model_files = sorted(p for p in args.model_dir.iterdir() if p.is_file())
    report = {"experiment": args.experiment, "train_file": args.train_file, "smoke_only": args.max_steps > 0,
              "train_rows": len(train), "dev_rows": len(dev), "before_dev": before, "after_dev": predicted.metrics,
              "dev_per_label": classification_report(dev.labels, guesses, labels=[0, 1, 2], target_names=LABELS, output_dict=True, zero_division=0),
              "truncated_pairs": truncation, "selection": "gold-evidence DEV Macro-F1; end-to-end promotion NOT evaluated",
              "training": {"lr": 1e-5, "epochs_max": args.epochs, "batch": 8, "gradient_accumulation": 2, "max_length": 256, "seed": 42, "ai_weight": 0.5},
              "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
              "bundle_manifest_sha256": checksum(manifest_path), "initial_model_files": {p.name: checksum(p) for p in model_files},
              "best_model_files": {p.name: checksum(p) for p in destination.iterdir() if p.is_file()},
              "warning": "AI silver is not human gold; no test read, no claim of real-world 80%; recalibrate and rebuild aggregator for new logits"}
    (args.output_dir / "finetune_metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(report, ensure_ascii=True, indent=2))


run_finetune()

